# Deep Learning 019 — Memoization & Backpropagation

Companion notebook to the lesson. Two ideas that turn out to be the same idea:

1. **Memoization** — if a computation reappears, store the answer instead of redoing it.
2. **Backpropagation with more than one hidden layer** — the derivative for an early weight
   contains, as a sub-expression, the entire derivative for a later one.

Backpropagation *is* memoized differentiation. That is the whole lesson, and the Fibonacci
example makes the size of the saving concrete: at `n = 40`, **331,160,281 calls versus 79**.

In [ ]:
import sys
import time
import numpy as np

sys.setrecursionlimit(10_000)

## Part A — Fibonacci, the standard demonstration

The naive recursion re-solves the same subproblems over and over. Count the calls rather
than guessing at the cost.

In [ ]:
calls = 0
def fib_naive(n):
    global calls
    calls += 1
    return n if n < 2 else fib_naive(n - 1) + fib_naive(n - 2)

print(f"{'n':>4}{'fib(n)':>10}{'calls':>12}{'calls / n':>12}")
for n in (5, 10, 20, 25, 30):
    calls = 0
    v = fib_naive(n)
    print(f"{n:>4}{v:>10}{calls:>12,}{calls / n:>12,.0f}")

In [ ]:
memo = {}
mcalls = 0
def fib_memo(n):
    global mcalls
    mcalls += 1
    if n < 2:
        return n
    if n not in memo:
        memo[n] = fib_memo(n - 1) + fib_memo(n - 2)
    return memo[n]

print(f"{'n':>4}{'fib(n)':>26}{'calls':>10}")
for n in (5, 10, 20, 30, 40, 80):
    memo, mcalls = {}, 0
    v = fib_memo(n)
    print(f"{n:>4}{v:>26,}{mcalls:>10}")

Naive is exponential; memoized is linear. Note the base, because it is a detail worth
getting right: the call tree is **not** a full binary tree — the `fib(n-2)` branch is one
level shorter than the `fib(n-1)` branch — so the count grows as the golden ratio to the
`n`, not 2 to the `n`. The exact count is `2*F(n+1) - 1`, which the next cell checks
against the measured numbers above.

In [ ]:
def fib(n):
    a, b = 0, 1
    for _ in range(n):
        a, b = b, a + b
    return a

def naive_calls(n):
    return 2 * fib(n + 1) - 1

# check the formula against the counts measured above
for n, measured in ((5, 15), (10, 177), (20, 21891), (25, 242785), (30, 2692537)):
    assert naive_calls(n) == measured, n
print("2*F(n+1) - 1 reproduces every measured call count exactly\n")

print(f"{'n':>4}{'naive calls':>26}{'memoized':>11}{'ratio':>20}")
for n in (30, 40, 60, 80):
    nc, mc = naive_calls(n), 2 * n - 1
    print(f"{n:>4}{nc:>26,}{mc:>11}{nc / mc:>24,.0f}x")

t0 = time.perf_counter(); calls = 0; fib_naive(28); t_naive = time.perf_counter() - t0
memo = {}; t0 = time.perf_counter(); fib_memo(28); t_memo = time.perf_counter() - t0
print(f"\nmeasured at n = 28: naive {t_naive * 1e3:.1f} ms, memoized {t_memo * 1e6:.0f} us"
      f"  ({t_naive / max(t_memo, 1e-9):,.0f}x)")

## Part B — The same waste appears in derivatives

Take a chain: input → hidden 1 → hidden 2 → hidden 3 → output, one node per layer, each
one $a_i = f(w_i a_{i-1})$. Write out the derivative of the loss with respect to each
weight and the repetition is obvious.

$$\frac{\partial L}{\partial w_3} = \frac{\partial L}{\partial a_3}\cdot\frac{\partial a_3}{\partial w_3}$$
$$\frac{\partial L}{\partial w_2} = \underbrace{\frac{\partial L}{\partial a_3}\cdot\frac{\partial a_3}{\partial a_2}}_{\text{reuses the above}}\cdot\frac{\partial a_2}{\partial w_2}$$
$$\frac{\partial L}{\partial w_1} = \underbrace{\frac{\partial L}{\partial a_3}\cdot\frac{\partial a_3}{\partial a_2}\cdot\frac{\partial a_2}{\partial a_1}}_{\text{reuses the above}}\cdot\frac{\partial a_1}{\partial w_1}$$

Each line is the previous line's prefix times one new factor. Compute them front to back
and you redo the prefix every time; compute them **back to front** and each prefix is
already in hand. That stored prefix has a name — it is the *delta* in every backprop
implementation you will ever read.

In [ ]:
def sigmoid(z):   return 1 / (1 + np.exp(-z))
def d_sigmoid(z): s = sigmoid(z); return s * (1 - s)

x, y = 0.7, 1.0
w = np.array([0.5, -0.8, 1.2])         # three weights, one per layer

def forward(w):
    a = [x]
    z = []
    for wi in w:
        zi = wi * a[-1]; z.append(zi); a.append(sigmoid(zi))
    return z, a                        # a[0] = x, a[-1] = prediction

def L(w):
    return (y - forward(w)[1][-1]) ** 2

### The wasteful way — recompute the whole chain for every weight

In [ ]:
mults = 0
def grad_naive(w):
    global mults
    z, a = forward(w)
    g = np.zeros(3)
    for i in range(3):                      # for each weight, walk the chain from the end
        prefix = -2 * (y - a[-1])
        mults += 1
        for k in range(2, i, -1):           # d a_{k+1} / d a_k, one factor per layer above i
            prefix *= d_sigmoid(z[k]) * w[k]
            mults += 2
        g[i] = prefix * d_sigmoid(z[i]) * a[i]
        mults += 2
    return g

mults = 0
gn = grad_naive(w)
naive_mults = mults
print("gradient:", np.round(gn, 6), f"   multiplications: {naive_mults}")

### The memoized way — carry the prefix backwards, computing each factor once

In [ ]:
def grad_memo(w):
    z, a = forward(w)
    g = np.zeros(3)
    delta = -2 * (y - a[-1])                # <- the memo. Everything reuses it.
    n = 1
    for i in range(2, -1, -1):
        g[i] = delta * d_sigmoid(z[i]) * a[i]
        n += 2
        if i > 0:
            delta = delta * d_sigmoid(z[i]) * w[i]
            n += 2
    return g, n

gm, memo_mults = grad_memo(w)
print("gradient:", np.round(gm, 6), f"   multiplications: {memo_mults}")
print("same answer:", np.allclose(gn, gm))

In [ ]:
# and check both against the definition of a derivative
h = 1e-7
num = np.array([(L(w + h * np.eye(3)[i]) - L(w - h * np.eye(3)[i])) / (2 * h) for i in range(3)])
print("numeric :", np.round(num, 6))
print("max error:", np.abs(num - gm).max())
assert np.abs(num - gm).max() < 1e-6

Three layers is too small for the saving to look impressive. Scale the depth and the two
costs separate the way Fibonacci's did — the naive version is quadratic in depth, the
memoized one linear.

In [ ]:
def count_ops(depth):
    naive = sum(1 + 2 * max(0, depth - 1 - i) + 2 for i in range(depth))
    memo = 1 + sum(2 + (2 if i > 0 else 0) for i in range(depth))
    return naive, memo

print(f"{'depth':>7}{'naive mults':>14}{'memoized':>11}{'ratio':>9}")
for d in (3, 10, 50, 200, 1000):
    n_, m_ = count_ops(d)
    print(f"{d:>7}{n_:>14,}{m_:>11,}{n_ / m_:>9.1f}x")

## Part C — Multi-path: when a node feeds more than one node

Real layers are not chains. A hidden node feeds *every* node in the next layer, so a change
in it reaches the loss along several routes. The multivariable chain rule says: **sum over
the paths.**

$$\frac{\partial L}{\partial a} = \sum_{k}\frac{\partial L}{\partial b_k}\cdot\frac{\partial b_k}{\partial a}$$

Memoization still applies, and it matters more — every $\partial L/\partial b_k$ is already
stored from the layer above, so the sum costs one dot product rather than one full re-walk
per path.

In [ ]:
r = np.random.default_rng(0)
W1 = r.normal(size=(3, 4)) * 0.5      # 3 inputs -> 4 hidden
W2 = r.normal(size=(4, 2)) * 0.5      # 4 hidden -> 2 outputs
xv = r.normal(size=3)
yv = np.array([1.0, 0.0])

def net(W1, W2):
    z1 = xv @ W1; a1 = sigmoid(z1)
    z2 = a1 @ W2; a2 = sigmoid(z2)
    return z1, a1, z2, a2

def loss(W1, W2):
    return float(((yv - net(W1, W2)[3]) ** 2).sum())

z1, a1, z2, a2 = net(W1, W2)
d2 = -2 * (yv - a2) * d_sigmoid(z2)          # memo for the output layer
gW2 = np.outer(a1, d2)
d1 = (d2 @ W2.T) * d_sigmoid(z1)             # <- the SUM OVER PATHS, as one dot product
gW1 = np.outer(xv, d1)

def numeric(W, other, which):
    g = np.zeros_like(W)
    for i in np.ndindex(W.shape):
        old = W[i]
        W[i] = old + 1e-6; up = loss(W1, W2)
        W[i] = old - 1e-6; dn = loss(W1, W2)
        W[i] = old
        g[i] = (up - dn) / 2e-6
    return g

print("max error on W2:", np.abs(numeric(W2, W1, 2) - gW2).max())
print("max error on W1:", np.abs(numeric(W1, W2, 1) - gW1).max())
print("\nd2 @ W2.T does the path sum for all 4 hidden nodes at once:", d1.shape)

`d2 @ W2.T` is the entire multivariable chain rule for that layer. Four hidden nodes, two
paths each, eight partial products — and it is one matrix multiply because the pieces were
all computed once and kept.

## The connection, stated plainly

| Fibonacci | Backpropagation |
|---|---|
| `fib(n-1)` and `fib(n-2)` overlap | the derivative for `w_i` contains the one for `w_{i+1}` |
| store results in a dict | store the *delta* for each layer |
| exponential → linear | quadratic in depth → linear in depth |
| recursion order matters | **backwards** is the order that makes the reuse possible |

Backpropagation is called *back*propagation for exactly this reason. Going forwards would
give the same numbers and throw away every intermediate result the next step needs.

## Try it yourself

1. Add a `print` inside `fib_naive` for `n = 5` and count how many times `fib(2)` is
   evaluated. Predict the number first from the recursion tree.
2. Extend `grad_memo` to four and five layers and confirm the operation count matches
   `count_ops`.
3. Break the memoization deliberately: recompute `delta` from scratch inside the loop. The
   gradient must be identical and the cost must not be. Verify both.
4. In Part C, replace `d2 @ W2.T` with an explicit double loop over paths. Same answer,
   and now you can see the sum the matrix multiply was hiding.